# Confounder problem — хто більше п'є пива в барі

**Питання, з якого починаємо:** чи справді старші люди п'ють більше пива?

**Гіпотеза, яку симулюємо на synthetic-даних:**
- Молодь 18–24 — переважно студенти / entry-level, часто без грошей на бар.
- Старші 25–50 мають стабільний дохід і ходять у бари вільно.
- Пиво п'ється переважно в барах, тому `bar_visits` → `beer` — майже пряма залежність.

**Що очікуємо побачити:**
- `corr(age, beer)` — помірна додатна, бо старші п'ють більше.
- Але це через `bar_visits` як confounder (грошовий тригер).
- Партиальна кореляція `age ↔ beer | control bar_visits` → падає до ~0.

Це той самий паттерн що з `fare / pclass / survived` на Титаніку — не саме `age` спричиняє більше пива, а те, що корелює з `age` (фінансова стабільність → доступ до бару).

## 1. Імпорти

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

rng = np.random.default_rng(42)
N = 200

## 2. Генеруємо synthetic-дані

Ланцюг генерації побудований так, щоб `bar_visits` був **справжнім** драйвером кількості пива, а `age` впливав лише опосередковано — через фінансовий стан:

```
age → no_money → bar_visits → beer_liters
```

Ключовий трюк — `no_money` різко зміщений для молоді (90% без грошей) і рідкісний у старших (5%). Далі `bar_visits` — Пуассонівський процес з середнім 0.3 (нема грошей) або 4.0 (є гроші).

In [ ]:
# Вік від 18 до 50
age = rng.integers(18, 51, size=N)

# Молодь без стабільного доходу: 18-24 роки
young_no_income = age < 25

# Відсутність грошей на регулярні відвідування бару (bool → 0/1 для кореляції)
no_money = np.where(
    young_no_income,
    rng.binomial(1, 0.9, size=N),   # 90% молодих фінансово обмежені
    rng.binomial(1, 0.05, size=N),  # 5% старших — форс-мажори
)

# Візити в бар за місяць: нема грошей → майже не ходить; є гроші → регулярно
bar_visits = np.where(
    no_money.astype(bool),
    rng.poisson(0.3, size=N),
    rng.poisson(4.0, size=N),
)

# Пиво (літри на місяць) = прямо пропорційне візитам + шум
beer_liters = bar_visits * 1.5 + rng.normal(0, 0.5, size=N)
beer_liters = np.clip(beer_liters, 0, None)  # не буває від'ємного

## 3. Складаємо DataFrame

In [ ]:
df = pd.DataFrame({
    "age": age,
    "no_money": no_money,
    "bar_visits": bar_visits,
    "beer_liters": beer_liters,
})

df.head(10)

## 4. Кореляційна матриця (сирий погляд)

Тут ми ще нічого не контролюємо — просто дивимось попарні Пірсони. Це те, що зазвичай кажуть у постах "глянь на кореляції першими!" — і тут і починається confounder-пастка.

In [ ]:
corr = df.corr()
corr.round(2)

In [ ]:
sns.heatmap(corr, annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Кореляції: вік, no_money, bar_visits, beer_liters")
plt.tight_layout()
plt.show()

## 5. Партиальна кореляція — вирахуємо вплив `bar_visits`

Формула:

$$
r_{XY|Z} = \frac{r_{XY} - r_{XZ} \cdot r_{YZ}}{\sqrt{(1 - r_{XZ}^2)(1 - r_{YZ}^2)}}
$$

де $Z$ = `bar_visits` (кандидат у confounder-и), $X$ = `age`, $Y$ = `beer_liters`.

**Читання формули:** з сирого зв'язку $r_{XY}$ відрізаємо частину, яку можна пояснити через $Z$. Знаменник — нормалізація, щоб результат залишався у діапазоні $[-1, 1]$.

In [ ]:
r_age_beer = df["age"].corr(df["beer_liters"])
r_age_bar = df["age"].corr(df["bar_visits"])
r_beer_bar = df["beer_liters"].corr(df["bar_visits"])

partial = (r_age_beer - r_age_bar * r_beer_bar) / np.sqrt(
    (1 - r_age_bar**2) * (1 - r_beer_bar**2)
)

print(f"corr(age, beer)                      = {r_age_beer:.3f}")
print(f"corr(age, beer | control bar_visits) = {partial:.3f}")

**Інтерпретація:**

- Перше число — сирий зв'язок `age ↔ beer` (помірний додатний).
- Друге число — зв'язок **після** вирахування впливу `bar_visits`.
- Якщо друге падає до ~0 → `age` впливає на `beer` **тільки** через візити в бар.
- Тобто `bar_visits` = confounder, а справжня причина — відсутність грошей у молоді → менше візитів → менше пива.

## 6. Візуалізація confounder-у

Дивимось на те саме інакше: якщо розбити людей на групи з **однаковою** частотою відвідування бару, то всередині кожної групи зв'язок `age ↔ beer` має зникнути. Це і є "контроль confounder-у" зроблений руками.

In [ ]:
df["bar_group"] = pd.cut(
    df["bar_visits"],
    bins=[-1, 1, 3, 20],
    labels=["Low (0-1)", "Mid (2-3)", "High (4+)"],
)

for name, subset in df.groupby("bar_group", observed=True):
    if len(subset) > 5:
        r = subset["age"].corr(subset["beer_liters"])
        print(f"  {name} (n={len(subset)}): corr = {r:+.3f}")

### Сирий scatter — видно висхідну хмару

In [ ]:
sns.regplot(
    data=df,
    x="age",
    y="beer_liters",
    scatter_kws={"alpha": 0.5},
    line_kws={"color": "red"},
)
plt.title(f"Сирий scatter age ↔ beer\n(r = {r_age_beer:.2f}, є нахил)")
plt.tight_layout()
plt.show()

### Всередині кожної групи `bar_visits` — нахил зникає

In [ ]:
g = sns.lmplot(
    data=df,
    x="age",
    y="beer_liters",
    col="bar_group",
    height=4,
    aspect=0.9,
    scatter_kws={"alpha": 0.6},
    line_kws={"color": "red"},
)
g.fig.suptitle(
    "Всередині кожної групи bar_visits залежність age → beer зникає\n"
    "(суть партиальної кореляції: контролюємо confounder → сигнал падає)",
    y=1.08,
)
plt.tight_layout()
plt.show()

## Висновок

Сирий Пірсон між `age` і `beer_liters` показав додатну кореляцію — на око здається, що з віком п'ють більше. Але:

1. **Партиальна кореляція** з контролем `bar_visits` падає до майже нуля.
2. **Стратифікація** (кореляція всередині груп однакового `bar_visits`) підтверджує те саме: нахилу немає.

Обидва методи говорять одне: `age` не впливає на `beer` напряму. Впливає `bar_visits`, а `age` просто корелює з ним через `no_money`.

**Правило на життя:** якщо два феномени скорельовані — перше питання не *"який напрямок причинності?"*, а *"а що ще з ними обома корелює?"*. Це і є confounder-у-голову підхід. У ML це важливо коли будуєш features: якщо додаси `age` як прямий предиктор — модель "вивчить" фейковий сигнал і зламається на нових даних, де розподіл `no_money` за віком інший.